# Lab 28 Kaggle Compat Server

Use this notebook if `vLLM` exits on Kaggle.

It starts one FastAPI server that exposes:

- `/v1/models`
- `/v1/chat/completions`
- `/embed`

The local lab can use the same public ngrok URL for both `VLLM_NGROK_URL` and `EMBED_NGROK_URL`.


In [ ]:
!pip install -q fastapi uvicorn pyngrok sentence-transformers transformers accelerate torch httpx requests

In [ ]:
from getpass import getpass

NGROK_AUTH_TOKEN = getpass("Paste NGROK_AUTH_TOKEN: ").strip()
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
SERVER_PORT = 8000
MAX_NEW_TOKENS = 192

assert NGROK_AUTH_TOKEN, "Fill in NGROK_AUTH_TOKEN first"

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print("Models loaded")

In [ ]:
import threading
from types import SimpleNamespace

import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = FastAPI(title="Lab28 Compat Server")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[ChatMessage]
    max_tokens: int | None = None

@app.get("/healthz")
def healthz():
    return {"status": "ok", "model": MODEL_NAME}

@app.get("/v1/models")
def list_models():
    return {
        "object": "list",
        "data": [
            {
                "id": MODEL_NAME,
                "object": "model",
                "owned_by": "kaggle-compat"
            }
        ]
    }

@app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    embeddings = embed_model.encode(texts).tolist()
    return {"embeddings": embeddings}

@app.post("/v1/chat/completions")
def chat_completions(payload: ChatRequest):
    messages = [m.model_dump() for m in payload.messages]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    generated = llm_model.generate(
        **inputs,
        max_new_tokens=payload.max_tokens or MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = generated[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return {
        "id": "chatcmpl-lab28",
        "object": "chat.completion",
        "model": MODEL_NAME,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop"
            }
        ]
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=SERVER_PORT)

threading.Thread(target=run_server, daemon=True).start()
gateway_tunnel = ngrok.connect(SERVER_PORT, "http")
print("PUBLIC_GATEWAY_URL=", gateway_tunnel.public_url)

In [ ]:
import requests

headers = {"ngrok-skip-browser-warning": "true"}

models_resp = requests.get(
    f"{gateway_tunnel.public_url}/v1/models",
    headers=headers,
    timeout=30,
)
print("Models status:", models_resp.status_code)
print(models_resp.text[:300])

embed_resp = requests.post(
    f"{gateway_tunnel.public_url}/embed",
    json={"texts": ["hello from lab 28"]},
    headers=headers,
    timeout=30,
)
print("Embed status:", embed_resp.status_code)
print("Embedding size:", len(embed_resp.json()["embeddings"][0]))

chat_resp = requests.post(
    f"{gateway_tunnel.public_url}/v1/chat/completions",
    json={
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": "Say hello from Kaggle in one sentence."}
        ]
    },
    headers=headers,
    timeout=120,
)
print("Chat status:", chat_resp.status_code)
print(chat_resp.text[:500])

In [ ]:
print("Copy these into your local .env file:")
print(f"VLLM_NGROK_URL={gateway_tunnel.public_url}")
print(f"EMBED_NGROK_URL={gateway_tunnel.public_url}")